#tO RUN BEFORE :
litellm --config litellm_config.yaml --port 4000

curl http://localhost:4000/v1/models


In [12]:
import json
import glob
import os
import unicodedata
from difflib import get_close_matches
from openai import OpenAI
from IPython.display import display, Markdown

# 1. SETUP OLLAMA
client = OpenAI(
    base_url="http://localhost:4000/v1",
    api_key="ollama"
)
MODEL = "local"

In [13]:


# 1. DÉFINITION DES SYNONYMES 
RISQUES_MAPPING = {
    "inondation": ["inondation", "crue", "submersion", "eau"],
    "mouvement de terrain": ["mouvement de terrain", "effondrement", "glissement"],
    "séisme": ["séisme", "sismique", "tremblement"],
    "feu": ["feu", "incendie", "forêt"],
    "tempête": ["tempête", "vent", "cyclone"],
    "industriel": ["industriel", "seveso", "usine", "chimique"],
    "nucléaire": ["nucléaire", "radioactif", "centrale"],
    "transport": ["transport", "matières dangereuses", "tmoy"]
}

# 2. CHARGEMENT DE L'INDEX (Indispensable pour avoir des données)
def load_indexes():
    indexes = {}
    # Adaptez le chemin si besoin (ex: "data/index/*.json" ou "indexer/data/index/*.json")
    files = glob.glob("indexer/data/index/*.json") + glob.glob("data/index/*.json")
    
    print(f"📂 Chargement des index depuis : {files}")
    for file_path in files:
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                indexes.update(data)
        except Exception as e:
            print(f"⚠️ Erreur lecture {file_path}: {e}")
            
    print(f"✅ Base chargée : {len(indexes)} entrées (Villes + Global).")
    return indexes

# On charge la base une seule fois
if 'INDEX_DB' not in globals():
    INDEX_DB = load_indexes()
    print(INDEX_DB)
else:
    print("⚡ Index déjà chargé.")
    print(INDEX_DB)

⚡ Index déjà chargé.
{'Aix-les-Bains': {'risques': ['Inondation'], 'refs': [{'section': '3.2. La Directive Européenne Inondation', 'type': 'texte', 'texte': "En Savoie, dans le cadre de l'application de la Directive inondation, le Préfet a chargé la  Direction  Départementale des Territoires (DDT), par arrêté du 20 juillet 2016, de coordonner l'élaboration des Stratégies Locales de Gestion des Risques d'Inondation (SLGRI) en Savoie et a arrêté la liste des parties prenantes, respectivement pour le Territoire à Risques importants d'Inondation (TRI) d'Albertville (14 communes) et le Territoire à Risques importants d'Inondation (TRI) de Chambéry - Aix-les-Bains (31 communes).", 'path': ['sous_sections', 77, 'contenu', 3], 'risque': 'Inondation'}]}, 'Albertville': {'risques': ['Inondation'], 'refs': [{'section': '3.2. La Directive Européenne Inondation', 'type': 'texte', 'texte': "En Savoie, dans le cadre de l'application de la Directive inondation, le Préfet a chargé la  Direction  Départ

In [14]:
# 2. DÉFINITION DE L'OUTIL
tools_schema = [{
    "type": "function",
    "function": {
        "name": "retrieve_from_index",
        "description": "Trouver les risques locaux et consignes de sécurité.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Type de risque (ex: inondation)"},
                "city": {"type": "string", "description": "Ville cible (ex: Poitiers)"}
            },
            "required": ["query"]
        }
    }
}]

In [15]:
import json, unicodedata
from difflib import get_close_matches

def remove_accents(input_str):
    if not isinstance(input_str, str):
        return ""
    nfkd_form = unicodedata.normalize("NFKD", input_str)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)]).lower()

def detect_risk_token(query_norm: str) -> str:
    for key, syns in RISQUES_MAPPING.items():
        if key in query_norm:
            print(f"[DEBUG] Risk matched by key: '{key}'")
            return key
        for s in syns:
            if s in query_norm:
                print(f"[DEBUG] Risk matched by synonym: '{s}' (group: {key})")
                return s
    print("[DEBUG] No risk detected → using full query")
    return query_norm

def retrieve_from_index(query, city=None):
    print("\n================ RETRIEVE START ================")
    print(f"[DEBUG] Raw query: {query}")
    print(f"[DEBUG] Raw city: {city}")

    results = []
    query_norm = remove_accents(query)
    print(f"[DEBUG] Normalized query: {query_norm}")

    detected_token = detect_risk_token(query_norm)
    print(f"[DEBUG] Detected risk token: {detected_token}")

    # -------- City detection --------
    target_city = None
    city_found = False

    if city:
        city_norm = remove_accents(city)
        print(f"[DEBUG] Normalized city: {city_norm}")

        all_cities = [k for k in INDEX_DB.keys() if k != "_GLOBAL_"]
        target_city = next(
            (c for c in all_cities if remove_accents(c) == city_norm),
            None
        )

        if not target_city:
            matches = get_close_matches(city, all_cities, n=1, cutoff=0.6)
            target_city = matches[0] if matches else None
            print(f"[DEBUG] Fuzzy city match: {matches}")

    print(f"[DEBUG] Target city resolved to: {target_city}")

    # -------- Local retrieval --------
    if target_city and target_city in INDEX_DB:
        print(f"[DEBUG] Searching LOCAL index for city: {target_city}")

        for ref in INDEX_DB[target_city]["refs"]:
            texte_norm = remove_accents(ref.get("texte", ""))
            risque_norm = remove_accents(ref.get("risque", ""))

            if detected_token in risque_norm or detected_token in texte_norm:
                section = ref.get("section", "Info Locale")
                print(f"[DEBUG] ✔ Local match in section '{section}'")
                results.append(
                    f"✅ [SOURCE: {target_city} | SECTION: {section}] {ref.get('texte','')}"
                )
                city_found = True

        print(f"[DEBUG] Local matches found: {len(results)}")

    # -------- Kill switch --------
    if city and not city_found:
        print("[DEBUG] ❌ City requested but NO local info found → KILL SWITCH")
        print("================ RETRIEVE END =================\n")
        return json.dumps([], ensure_ascii=False)

    # -------- Global retrieval --------
    if "_GLOBAL_" in INDEX_DB:
        print("[DEBUG] Searching GLOBAL index")

        before_global = len(results)
        for ref in INDEX_DB["_GLOBAL_"]["refs"]:
            texte_norm = remove_accents(ref.get("texte", ""))
            risque_norm = remove_accents(ref.get("risque", ""))

            if detected_token in risque_norm or detected_token in texte_norm:
                section = ref.get("section", "Consigne")
                print(f"[DEBUG] ✔ Global match in section '{section}'")
                results.append(
                    f"✅ [SOURCE: National | SECTION: {section}] {ref.get('texte','')}"
                )

        print(f"[DEBUG] Global matches added: {len(results) - before_global}")

    print(f"[DEBUG] TOTAL results returned: {len(results[:8])}")
    print("================ RETRIEVE END =================\n")

    return json.dumps(results[:8], ensure_ascii=False)


In [16]:
# 4. AGENT LOOP
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "retrieve_from_index",
            "description": "Fetch official safety instructions and local risks from the Index.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The risk (e.g., 'inondation', 'feu')"},
                    "city": {"type": "string", "description": "Target city (e.g., 'Poitiers'). Optional."}
                },
                "required": ["query"]
            }
        }
    }
]


In [17]:
def ask_agent(user_question):
    # 1. PROMPT SYSTÈME AJUSTÉ (Autorise à sauter des sections vides)
    system_prompt = """
    Tu es un expert en sécurité civile strict.
    
    ### PROTOCOLE DE RÉPONSE :
    1. **Squelette Flexible** : Essaie de remplir ces 3 sections, MAIS SI TU N'AS PAS L'INFO, NE L'ÉCRIS PAS :
       - 🏘️ **Situation Locale** (Uniquement si info disponible)
       - 📜 **Historique** (Uniquement si info disponible)
       - 🚨 **Consignes de Sécurité** (Obligatoire)
    
    2. **ANTI-HALLUCINATION ABSOLUE** :
       - Tu ne dois utiliser QUE le texte fourni par l'outil.
       - Chaque phrase doit avoir une source : (Source : Poitiers | Section : Alerte).
       - Si l'outil est vide, dis "Je ne sais pas".
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question}
    ]
    
    print("🤖 1. L'agent réfléchit...")
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools_schema,
        # 👇 On l'oblige à utiliser l'outil
        tool_choice={"type": "function", "function": {"name": "retrieve_from_index"}}
    )    
    msg = response.choices[0].message
    print(msg.tool_calls)
    if msg.tool_calls:
        messages.append(msg)
        for tool in msg.tool_calls:
            if tool.function.name == "retrieve_from_index":
                args = json.loads(tool.function.arguments)
                print(f"🔎 2. Recherche : {args}")
                
                evidence_json = retrieve_from_index(args["query"], args.get("city"))
                
                # --- LE COUPE-CIRCUIT (C'EST ICI QUE TU RÈGLES LE PROBLÈME) ---
                evidence = json.loads(evidence_json)
                
                if not evidence:
                    print("⚠️ 3. ALERTE : Index vide ! Arrêt d'urgence.")
                    # On ne laisse même pas le modèle essayer de répondre.
                    return "❌ **Information introuvable.** \n\nJe n'ai trouvé aucune information pertinente dans les documents officiels pour cette demande. Je ne peux pas inventer de réponse."
                
                # Si on a des preuves, on continue
                print(f"✅ 3. Trouvé {len(evidence)} éléments.")
                messages.append({"role": "tool", "tool_call_id": tool.id, "content": evidence_json})
        
        print("📝 4. Rédaction...")
        final = client.chat.completions.create(model=MODEL, messages=messages)
        return final.choices[0].message.content
    
    return msg.content

In [ ]:
# --- TEST ---
print("--- TEST: Inondation à Poitiers ---")
answer = ask_agent("Quels sont les risques d'inondation à Poitiers ?")
display(Markdown(answer))

--- TEST: Inondation à Poitiers ---
🤖 1. L'agent réfléchit...
[ChatCompletionMessageFunctionToolCall(id='call_0445260c-5c0e-4c51-ae29-9f4c93faf373', function=Function(arguments='{"query": "inondation", "city": "Poitiers"}', name='retrieve_from_index'), type='function')]
🔎 2. Recherche : {'query': 'inondation', 'city': 'Poitiers'}

================ RETRIEVE START ================
[DEBUG] Raw query: inondation
[DEBUG] Raw city: Poitiers
[DEBUG] Normalized query: inondation
[DEBUG] Risk matched by key: 'inondation'
[DEBUG] Detected risk token: inondation
[DEBUG] Normalized city: poitiers
[DEBUG] Target city resolved to: Poitiers
[DEBUG] Searching LOCAL index for city: Poitiers
[DEBUG] ✔ Local match in section '4 - HISTORIQUE DES PRINCIPALES INONDATIONS DU DÉPARTEMENT'
[DEBUG] ✔ Local match in section 'OÙ S'INFORMER  SUR LE RISQUE INONDATION ?'
[DEBUG] Local matches found: 2
[DEBUG] Searching GLOBAL index
[DEBUG] ✔ Global match in section '6 - CONSIGNES INDIVIDUELLES DE SÉCURITÉ'
[DEBUG] G

^C
